# 03 — Componentes da série e estacionariedade

No notebook anterior, visualizamos a demanda de bicicletas ao longo do tempo.

Agora vamos observar melhor os componentes da série:

- tendência;
- sazonalidade;
- ruído;
- média móvel;
- estacionariedade.

A ideia é entender melhor o comportamento da série antes de partir para transformações e modelos.

### 03.01 Imports e caminhos

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

import plotly.graph_objects as go

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller

if Path.cwd().name == "notebooks":
    raiz_projeto = Path.cwd().parent
else:
    raiz_projeto = Path.cwd()

sys.path.append(str(raiz_projeto / "src"))

from visual_utils import (
    CORES,
    aplicar_layout_padrao,
    grafico_linha_padrao
)

caminho_serie_diaria = raiz_projeto / "outputs" / "tabelas" / "serie_diaria_bike.csv"
caminho_outputs_tabelas = raiz_projeto / "outputs" / "tabelas"

print("Série diária:")
print(caminho_serie_diaria)

### 03.02 Carregar série diária

In [ ]:
serie_diaria = pd.read_csv(caminho_serie_diaria)

serie_diaria["data_hora"] = pd.to_datetime(serie_diaria["data_hora"])

serie_diaria = serie_diaria.sort_values("data_hora").reset_index(drop=True)

print(f"Linhas: {serie_diaria.shape[0]}")
print(f"Colunas: {serie_diaria.shape[1]}")

serie_diaria.head()

## Série diária

Vamos começar retomando a série diária de demanda.

Essa visualização será nossa base para discutir tendência, sazonalidade e ruído.

### 03.03 Visualizar série diária

In [ ]:
fig = grafico_linha_padrao(
    df=serie_diaria,
    x="data_hora",
    y="demanda",
    titulo="Demanda diária de bicicletas",
    labels={
        "data_hora": "Data",
        "demanda": "Demanda diária"
    },
    altura=500
)

fig.show()

## Média móvel

A média móvel ajuda a suavizar oscilações da série.

Ela não é um modelo de previsão aqui. É apenas uma ferramenta visual para enxergar melhor o comportamento geral.

### 03.04 Criar média móvel

In [ ]:
serie_diaria["media_movel_7"] = (
    serie_diaria["demanda"]
    .rolling(window=7)
    .mean()
)

serie_diaria.head(10)

### 03.05 Série original versus média móvel

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=serie_diaria["data_hora"],
        y=serie_diaria["demanda"],
        mode="lines",
        name="Demanda diária",
        line=dict(color=CORES["azul_claro"], width=2)
    )
)

fig.add_trace(
    go.Scatter(
        x=serie_diaria["data_hora"],
        y=serie_diaria["media_movel_7"],
        mode="lines",
        name="Média móvel 7 períodos",
        line=dict(color=CORES["azul_principal"], width=3)
    )
)

fig = aplicar_layout_padrao(
    fig,
    titulo="Demanda diária e média móvel",
    altura=500
)

fig.show()

## Decomposição da série

Uma série temporal pode ser pensada como a combinação de três partes:

- tendência: movimento geral da série;
- sazonalidade: padrão que se repete;
- resíduo: parte irregular ou ruído.

Vamos usar uma decomposição aditiva para separar esses componentes.

### 03.06 Decompor a série

In [ ]:
serie_para_decompor = serie_diaria.set_index("data_hora")["demanda"]

decomposicao = seasonal_decompose(
    serie_para_decompor,
    model="additive",
    period=7
)

componentes = pd.DataFrame({
    "data_hora": serie_para_decompor.index,
    "observado": decomposicao.observed,
    "tendencia": decomposicao.trend,
    "sazonalidade": decomposicao.seasonal,
    "residuo": decomposicao.resid
}).reset_index(drop=True)

componentes.head()

### 03.07 Visualizar tendência

In [ ]:
fig = grafico_linha_padrao(
    df=componentes,
    x="data_hora",
    y="tendencia",
    titulo="Componente de tendência",
    labels={
        "data_hora": "Data",
        "tendencia": "Tendência"
    },
    altura=450
)

fig.show()

### 03.08 Visualizar sazonalidade

In [ ]:
fig = grafico_linha_padrao(
    df=componentes,
    x="data_hora",
    y="sazonalidade",
    titulo="Componente sazonal",
    labels={
        "data_hora": "Data",
        "sazonalidade": "Sazonalidade"
    },
    altura=450
)

fig.show()

### 03.09 Visualizar resíduo

In [ ]:
fig = grafico_linha_padrao(
    df=componentes,
    x="data_hora",
    y="residuo",
    titulo="Componente residual",
    labels={
        "data_hora": "Data",
        "residuo": "Resíduo"
    },
    altura=450
)

fig.show()

## Teste de estacionariedade

Modelos clássicos, como ARIMA, costumam funcionar melhor quando a série é estacionária.

Uma série estacionária tende a ter média, variância e dependência temporal mais estáveis ao longo do tempo.

Vamos aplicar o teste ADF para avaliar essa característica.

### 03.10 Função para teste ADF

In [ ]:
def teste_adf(serie):
    resultado = adfuller(serie.dropna())

    tabela_resultado = pd.DataFrame({
        "métrica": [
            "estatística_adf",
            "p_valor",
            "lags_utilizados",
            "observações"
        ],
        "valor": [
            resultado[0],
            resultado[1],
            resultado[2],
            resultado[3]
        ]
    })

    valores_criticos = pd.DataFrame({
        "métrica": [f"valor_crítico_{chave}" for chave in resultado[4].keys()],
        "valor": list(resultado[4].values())
    })

    return pd.concat([tabela_resultado, valores_criticos], ignore_index=True)

### 03.11 Aplicar teste ADF

In [ ]:
resultado_adf_original = teste_adf(serie_diaria["demanda"])

resultado_adf_original

### 03.12 Interpretação simples do p-valor

In [ ]:
p_valor = resultado_adf_original.loc[
    resultado_adf_original["métrica"] == "p_valor",
    "valor"
].iloc[0]

print(f"p-valor: {p_valor:.4f}")

if p_valor <= 0.05:
    print("Resultado: há evidência de estacionariedade.")
    print("A série original já apresenta um comportamento mais estável.")
else:
    print("Resultado: não há evidência suficiente de estacionariedade.")
    print("No próximo notebook, vamos aplicar diferenciação para tentar tornar a série mais estável.")

### 03.13 Salvar resultados

In [ ]:
caminho_componentes = caminho_outputs_tabelas / "componentes_serie_bike.csv"
caminho_adf = caminho_outputs_tabelas / "resultado_adf_original.csv"

componentes.to_csv(
    caminho_componentes,
    index=False,
    encoding="utf-8-sig"
)

resultado_adf_original.to_csv(
    caminho_adf,
    index=False,
    encoding="utf-8-sig"
)

print("Arquivos salvos:")
print("-", caminho_componentes)
print("-", caminho_adf)

## Próximo passo

Neste notebook, observamos tendência, sazonalidade, ruído e estacionariedade.

No próximo notebook, vamos trabalhar com diferenciação da série e entender como ACF e PACF ajudam na escolha dos parâmetros dos modelos.